# Χάρτης SOM (Self-Organizing Map)
Εκπαιδεύουμε έναν χάρτη Kohonen **μόνο με τα χαρακτηριστικά εισόδου** (χωρίς ετικέτες) και μετά βλέπουμε
πού «πέφτουν» οι κανονικές ροές και πού οι επιθέσεις.
- Αν οι επιθέσεις μαζεύονται σε συγκεκριμένες περιοχές του χάρτη → τα χαρακτηριστικά **σχετίζονται** με την ετικέτα.
- Αν είναι απλωμένες παντού με το ίδιο ποσοστό → **δεν σχετίζονται**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import chi2_contingency

DATA = "../data/"
FIG = "../figures/"
ROWS, COLS = 12, 12        # μέγεθος χάρτη (144 νευρώνες)
SEED = 42

## 1. Υλοποίηση SOM (batch αλγόριθμος, μόνο με numpy)
Κάθε νευρώνας του πλέγματος έχει ένα διάνυσμα βαρών ίδιου μεγέθους με την είσοδο. Σε κάθε εποχή:
1. κάθε δείγμα βρίσκει τον πιο κοντινό του νευρώνα (**BMU** – Best Matching Unit),
2. κάθε νευρώνας μετακινείται προς τον μέσο όρο των δειγμάτων που ανήκουν σε αυτόν **και στους γείτονές του** (γκαουσιανή γειτονιά),
3. η ακτίνα της γειτονιάς `sigma` μικραίνει σταδιακά.

Έτσι γειτονικοί νευρώνες καταλήγουν να αντιπροσωπεύουν παρόμοια δείγματα.

In [ ]:
class SOM:
    def __init__(self, rows, cols, n_epochs=40, sigma_end=0.6, seed=42):
        self.rows, self.cols, self.n_epochs, self.sigma_end, self.seed = rows, cols, n_epochs, sigma_end, seed
        r, c = np.meshgrid(np.arange(rows), np.arange(cols), indexing="ij")
        self.grid = np.column_stack([r.ravel(), c.ravel()]).astype("float32")      # θέση κάθε νευρώνα
        self.grid_d2 = ((self.grid[:, None, :] - self.grid[None, :, :]) ** 2).sum(-1)

    def _init_weights(self, X):
        # Γραμμική αρχικοποίηση πάνω στο επίπεδο των 2 πρώτων κύριων συνιστωσών (PCA)
        mean = X.mean(0)
        _, s, vt = np.linalg.svd(X[np.random.default_rng(self.seed).choice(len(X), min(len(X), 20000), replace=False)] - mean,
                                 full_matrices=False)
        sd = s[:2] / np.sqrt(min(len(X), 20000))
        a = np.linspace(-2, 2, self.rows)[self.grid[:, 0].astype(int)]
        b = np.linspace(-2, 2, self.cols)[self.grid[:, 1].astype(int)]
        return (mean + np.outer(a * sd[0], vt[0]) + np.outer(b * sd[1], vt[1])).astype("float32")

    def bmu(self, X, chunk=20000):
        out = np.empty(len(X), dtype=np.int64)
        w2 = (self.W ** 2).sum(1)
        for i in range(0, len(X), chunk):
            xb = X[i:i + chunk]
            out[i:i + chunk] = np.argmin(w2 - 2 * xb @ self.W.T, axis=1)
        return out

    def fit(self, X):
        X = np.asarray(X, dtype="float32")
        self.W = self._init_weights(X)
        sigma_start = max(self.rows, self.cols) / 2
        for t in range(self.n_epochs):
            sigma = sigma_start * (self.sigma_end / sigma_start) ** (t / (self.n_epochs - 1))
            b = self.bmu(X)
            sums = np.zeros_like(self.W); np.add.at(sums, b, X)
            counts = np.bincount(b, minlength=len(self.W)).astype("float32")
            H = np.exp(-self.grid_d2 / (2 * sigma ** 2)).astype("float32")
            self.W = (H @ sums) / (H @ counts)[:, None]
        return self

    def quantization_error(self, X):
        X = np.asarray(X, dtype="float32")
        return float(np.linalg.norm(X - self.W[self.bmu(X)], axis=1).mean())

    def u_matrix(self):
        # μέση απόσταση βαρών κάθε νευρώνα από τους (έως 8) γείτονές του
        Wg = self.W.reshape(self.rows, self.cols, -1)
        U = np.zeros((self.rows, self.cols))
        for r in range(self.rows):
            for c in range(self.cols):
                d = [np.linalg.norm(Wg[r, c] - Wg[rr, cc])
                     for rr in range(max(0, r - 1), min(self.rows, r + 2))
                     for cc in range(max(0, c - 1), min(self.cols, c + 2)) if (rr, cc) != (r, c)]
                U[r, c] = np.mean(d)
        return U

## 2. Συναρτήσεις σχεδίασης
- **Χάρτης Α (U-matrix):** χρώμα = μέση απόσταση από τους γείτονες. Σκούρο = ομοιογενής περιοχή (συστάδα), ανοιχτό = «σύνορο» ανάμεσα σε συστάδες. Μέσα σε κάθε κουτάκι: αριθμός συστάδας (C1…C144) και πλήθος δειγμάτων.
- **Χάρτης Β (ετικέτες):** χρώμα και τιμή = ποσοστό επιθέσεων στο κουτάκι, και πλήθος δειγμάτων.
- **Χάρτες χαρακτηριστικών (component planes):** η τιμή κάθε χαρακτηριστικού πάνω στον χάρτη. Χαρακτηριστικά με παρόμοια εικόνα είναι συσχετισμένα μεταξύ τους· αν κάποιο μοιάζει με τον χάρτη «% Attack», σχετίζεται με την ετικέτα.

In [ ]:
ATTACK_CMAP = LinearSegmentedColormap.from_list("normal_attack", ["#2a78d6", "#e9e9e6", "#eb6834"])

def _text_color(rgba):
    return "black" if 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2] > 0.45 else "white"

def _annotate(ax, im, texts, fontsize=6.5):
    rows, cols = texts.shape
    for r in range(rows):
        for c in range(cols):
            ax.text(c, r, texts[r, c], ha="center", va="center", fontsize=fontsize,
                    color=_text_color(im.cmap(im.norm(im.get_array()[r, c]))), linespacing=1.3)
    ax.set_xlabel("SOM column"); ax.set_ylabel("SOM row")
    ax.set_xticks(range(0, cols, 2)); ax.set_yticks(range(0, rows, 2)); ax.grid(False)

def cell_stats(som, X, is_attack):
    b = som.bmu(np.asarray(X, dtype="float32"))
    n = np.bincount(b, minlength=som.rows * som.cols)
    k = np.bincount(b, weights=is_attack, minlength=som.rows * som.cols)
    with np.errstate(invalid="ignore", divide="ignore"):
        rate = np.where(n > 0, k / n, np.nan)
    return n.reshape(som.rows, som.cols), k.reshape(som.rows, som.cols), rate.reshape(som.rows, som.cols)

def plot_umatrix(som, n, title, fname):
    U = som.u_matrix()
    fig, ax = plt.subplots(figsize=(10, 8.5))
    im = ax.imshow(U, cmap="viridis")
    ids = np.arange(1, n.size + 1).reshape(n.shape)
    _annotate(ax, im, np.array([[f"C{ids[r, c]}\n{n[r, c]}" for c in range(n.shape[1])] for r in range(n.shape[0])]))
    fig.colorbar(im, ax=ax, shrink=0.75, label="Mean neighbor distance")
    ax.set_title(f"{title}\nLabeled SOM – U-matrix (cluster ID and sample count)")
    plt.tight_layout(); plt.savefig(FIG + fname, dpi=200); plt.show()

def plot_attack_map(n, rate, title, fname, overall):
    fig, ax = plt.subplots(figsize=(10, 8.5))
    im = ax.imshow(np.nan_to_num(rate, nan=0.5) * 100, cmap=ATTACK_CMAP, vmin=0, vmax=100)
    txt = np.array([[f"{rate[r, c]:.0%}\nn={n[r, c]}" if n[r, c] > 0 else "–"
                     for c in range(n.shape[1])] for r in range(n.shape[0])])
    _annotate(ax, im, txt)
    fig.colorbar(im, ax=ax, shrink=0.75, label="% Attack στο κουτάκι  (μπλε = Normal, πορτοκαλί = Attack)")
    ax.set_title(f"{title}\nΠοσοστό επιθέσεων ανά κουτάκι (συνολικό ποσοστό: {overall:.1%})")
    plt.tight_layout(); plt.savefig(FIG + fname, dpi=200); plt.show()

def plot_planes(som, feature_names, planes, rate, title, fname):
    k = len(planes) + 1; ncol = 4; nrow = int(np.ceil(k / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.3 * nrow)); axes = axes.ravel()
    for ax, f in zip(axes, planes):
        im = ax.imshow(som.W[:, feature_names.index(f)].reshape(som.rows, som.cols), cmap="viridis")
        ax.set_title(f, fontsize=10); fig.colorbar(im, ax=ax, shrink=0.8)
    im = axes[len(planes)].imshow(np.nan_to_num(rate, nan=0.5) * 100, cmap=ATTACK_CMAP, vmin=0, vmax=100)
    axes[len(planes)].set_title("% Attack (ετικέτα)", fontsize=10, fontweight="bold"); fig.colorbar(im, ax=axes[len(planes)], shrink=0.8)
    for ax in axes: ax.set_xticks([]); ax.set_yticks([])
    for ax in axes[k:]: ax.axis("off")
    fig.suptitle(f"{title} – χάρτες χαρακτηριστικών (τυποποιημένες τιμές)", y=1.0)
    plt.tight_layout(); plt.savefig(FIG + fname, dpi=200); plt.show()

def report(n, k, rate, overall, name):
    m = n > 0
    purity = (np.maximum(k[m], n[m] - k[m]).sum()) / n.sum()
    chi2, p, dof, _ = chi2_contingency(np.vstack([k[m], n[m] - k[m]]))
    cramers_v = np.sqrt(chi2 / n.sum())
    print(f"--- {name} ---")
    print(f"Ποσοστό επιθέσεων ανά κουτάκι: ελάχ. {np.nanmin(rate):.1%} | μέγ. {np.nanmax(rate):.1%} | συνολικό {overall:.1%}")
    print(f"Purity (ακρίβεια αν κάθε κουτάκι πάρει την πλειοψηφούσα ετικέτα): {purity:.1%}"
          f"  [βάση = {max(overall, 1 - overall):.1%} αν λέμε πάντα την πλειοψηφούσα κλάση]")
    print(f"Έλεγχος χ² (κουτάκι × ετικέτα): Cramér's V = {cramers_v:.3f}, p-value = {p:.3g}")

## 3. SOM στο CTDAPD
Είσοδοι: οι 9 που είχαν οριστεί. Τα αριθμητικά τυποποιούνται (μέσος 0, τυπική απόκλιση 1), τα κατηγορικά γίνονται one-hot (0/1).
Η στήλη `Label` **δεν** δίνεται στον χάρτη· χρησιμοποιείται μόνο μετά, για να «χρωματίσουμε» τα κουτάκια.

In [ ]:
ct = pd.read_csv(DATA + "ctdapd_clean.csv", keep_default_na=False)

ct_num = ["Flow_Duration", "Packet_Size", "Total_Forward_Packets",
          "Total_Backward_Packets", "IDS_Alert_Count", "Anomaly_Score"]
ct_cat = ["Destination_Port", "Protocol_Type", "System_Patch_Status"]

Xn = (ct[ct_num] - ct[ct_num].mean()) / ct[ct_num].std()
Xc = pd.get_dummies(ct[ct_cat].astype(str), dtype="float32")
X_ct = pd.concat([Xn, Xc], axis=1)
y_ct = (ct["Label"] == "Attack").to_numpy(float)
ct_features = X_ct.columns.tolist()
print(X_ct.shape, "| ποσοστό επιθέσεων:", round(y_ct.mean(), 3))

som_ct = SOM(ROWS, COLS, seed=SEED).fit(X_ct.to_numpy())
print("Quantization error:", round(som_ct.quantization_error(X_ct.to_numpy()), 3))

In [ ]:
n_ct, k_ct, rate_ct = cell_stats(som_ct, X_ct.to_numpy(), y_ct)
plot_umatrix(som_ct, n_ct, "CTDAPD", "08_som_ctdapd_umatrix.png")
plot_attack_map(n_ct, rate_ct, "CTDAPD", "09_som_ctdapd_attack.png", y_ct.mean())
report(n_ct, k_ct, rate_ct, y_ct.mean(), "CTDAPD")

In [ ]:
plot_planes(som_ct, ct_features, ct_features, rate_ct, "CTDAPD", "10_som_ctdapd_planes.png")

## 4. SOM στο UNSW-NB15 (για σύγκριση)
Είσοδοι: τα 39 αριθμητικά χαρακτηριστικά ροής. Όσα έχουν πολύ ασύμμετρη κατανομή περνούν από `log(1+x)` και μετά όλα τυποποιούνται.

In [ ]:
un = pd.read_csv(DATA + "UNSW_NB15_train-set.csv")
un_num = [c for c in un.columns if c not in ["id", "proto", "service", "state", "attack_cat", "label"]]

Xu = un[un_num].astype("float64").copy()
skewed = Xu.skew()[lambda s: s > 2].index
Xu[skewed] = np.log1p(Xu[skewed])
Xu = (Xu - Xu.mean()) / Xu.std()
y_un = un["label"].to_numpy(float)
print(Xu.shape, "| ποσοστό επιθέσεων:", round(y_un.mean(), 3))

som_un = SOM(ROWS, COLS, seed=SEED).fit(Xu.to_numpy())
print("Quantization error:", round(som_un.quantization_error(Xu.to_numpy()), 3))

In [ ]:
n_un, k_un, rate_un = cell_stats(som_un, Xu.to_numpy(), y_un)
plot_umatrix(som_un, n_un, "UNSW-NB15", "11_som_unsw_umatrix.png")
plot_attack_map(n_un, rate_un, "UNSW-NB15", "12_som_unsw_attack.png", y_un.mean())
report(n_un, k_un, rate_un, y_un.mean(), "UNSW-NB15")

In [ ]:
unsw_planes = ["dur", "sbytes", "dbytes", "rate", "sttl", "dttl", "sload", "dload", "smean", "dmean", "ct_state_ttl"]
plot_planes(som_un, un_num, unsw_planes, rate_un, "UNSW-NB15", "13_som_unsw_planes.png")